# DyTopo — Real Dataset Evaluation

**Paper:** Lu et al. 2026 — *DyTopo: Dynamic Topology Routing for Multi-Agent Reasoning via Semantic Matching* (arXiv:2602.06039)

## Datasets
| Dataset | Domain | Difficulty | N | HuggingFace ID |
|---------|--------|-----------|---|----------------|
| HumanEval | Code | Fundamental | 164 | `openai/openai_humaneval` |
| APPS-Competition | Code | Advanced | 100 of 1000 | `codeparrot/apps` |
| MATH-500 | Math | Hard | 500 | `HuggingFaceH4/MATH-500` |
| Omni-MATH | Math | Expert | 70 (7×10) | `KbsdJames/Omni-MATH` |

# ── Vast.ai Setup Guide ───────────────────────────────────────────────────────

## Before running this notebook on Vast.ai

### 1. Rent the instance
- Template: **PyTorch (Vast)**
- GPU: **2× RTX PRO 6000** (or any 2× GPU with ≥180 GB total VRAM for 72B)
- Disk: **≥250 GB** (144 GB model + 9 GB venv + OS headroom)
- Enable: **SSH + Jupyter**

### 2. Add SSH key
Account → SSH Keys → paste your `~/.ssh/id_ed25519.pub`

### 3. Connect (from your Mac terminal)
```bash
ssh -p <PORT> root@<HOST> -i ~/.ssh/id_ed25519 -L 9999:localhost:8080
```

### 4. Start Jupyter on the instance (in SSH terminal)
```bash
jupyter lab --ip=0.0.0.0 --port=8080 --no-browser --allow-root
```
Connect VS Code to `http://localhost:9999?token=<token>`

### 5. Install ALL missing packages from SSH terminal (do NOT install vllm/torch — pre-installed)
```bash
/venv/main/bin/pip install \
  sentence-transformers datasets httpx nest_asyncio \
  scikit-learn huggingface_hub matplotlib pandas tqdm networkx
```
Or just run **Cell 2** in the notebook (it calls sys.executable so it targets the right venv).

### 6. (Fresh instance only) Install vLLM if missing
```bash
/venv/main/bin/pip install vllm
```

### 7. Start vLLM from SSH terminal (NOT from notebook)
```bash
HF_HOME=/root/.cache/huggingface \
/venv/main/bin/python -m vllm.entrypoints.openai.api_server \
  --model Qwen/Qwen2.5-72B-Instruct \
  --max-model-len 8192 \
  --tensor-parallel-size 2 \
  --port 8000 \
  --served-model-name dytopo-model \
  --trust-remote-code \
  --no-enable-log-requests \
  > /root/vllm.log 2>&1 &

tail -f /root/vllm.log
```
Wait until you see `Application startup complete` then run Cell 6 in the notebook.

> **Note on `--disable-log-requests`:** If you get an "unrecognized argument" error on a newer vLLM,
> replace it with `--no-enable-log-requests`.

### 8. Run notebook cells in order
Cell 1 → **Cell 2 (deps)** → Cell 3 → Cell 4 → Cell 5 → Cell 6 (wait for vLLM) → Cell 7 onwards.

### Cost estimate
| Phase | Time | Cost |
|---|---|---|
| Model download (144 GB) | ~25 min | ~$1.10 + $0.58 bandwidth |
| Full benchmark (72B, 3 datasets) | ~4–5 hrs | ~$13 |
| Stopped overnight | 8 hrs | ~$1.50 (disk only) |

> **Note:** Stop (not Destroy) the instance between sessions — model weights stay on disk.
> SSH port changes on restart.

In [1]:
!nvidia-smi
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free,compute_cap',
                         '--format=csv,noheader'], capture_output=True, text=True)
print('GPU summary:', result.stdout.strip())

import re
vram_match = re.search(r'(\d+) MiB', result.stdout)
if vram_match:
    vram_gb = int(vram_match.group(1)) / 1024
    print(f'\nVRAM: {vram_gb:.0f} GB')
    if vram_gb >= 80:
        print('G4 / RTX PRO 6000 confirmed — can run Qwen2.5-72B in BF16')
    elif vram_gb >= 24:
        print('L4 detected — will use Qwen2.5-7B')
    else:
        print('T4 detected (16 GB) — limited to 7B Q4 models')

Mon Oct 12 01:57:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 615.71.09              KMD Version: 615.71.09     CUDA UMD Version: 13.4     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    On  |   00000000:01:00.0 Off |                  Off |
| 30%   45C    P5             51W /  547W |       2MiB /  97887MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# ──get Install dependencies ──────────────────────────────────────
import sys, subprocess

pkgs = [
    'sentence-transformers',  # semantic embedding for adjacency routing
    'datasets',               # HuggingFace datasets (HumanEval, MATH-500, Omni-MATH)
    'httpx',                  # async HTTP client for vLLM API calls
    'nest_asyncio',           # allow await inside Jupyter's running event loop
    'scikit-learn',           # cosine_similarity fallback
    'huggingface_hub',        # hf_hub_download, snapshot_download
    'matplotlib',             # benchmark charts
    'pandas',                 # summary tables
    'tqdm',                   # progress bars with ETA
    'networkx',               # graph utilities (topology visualisation)
]

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q'] + pkgs,
    capture_output=True, text=True
)
if result.returncode == 0:
    print(f'✓ All {len(pkgs)} dependencies ready')
    print(f'  Python: {sys.executable}')
    print(f'  Packages: {", ".join(pkgs)}')
else:
    print(result.stderr[-800:])
    raise RuntimeError('pip install failed — see above')

In [2]:
import os
from pathlib import Path

RESULTS_DIR = Path('/workspace/results_v2')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Results directory: {RESULTS_DIR}')
print(f'Existing files: {list(RESULTS_DIR.iterdir())}')

Results directory: /workspace/results_v2
Existing files: [PosixPath('/workspace/results_v2/humaneval_results.json'), PosixPath('/workspace/results_v2/math500_results.json')]


In [3]:
import urllib.request, os, sys

ENGINE_URL  = 'https://raw.githubusercontent.com/51103220/agentic-dynamic-topology/refs/heads/main/api/dytopo_engine.py'
ENGINE_PATH = '/workspace/dytopo_engine.py'

try:
    urllib.request.urlretrieve(ENGINE_URL, ENGINE_PATH)
    print(f'Downloaded dytopo_engine.py ({os.path.getsize(ENGINE_PATH):,} bytes)')
except Exception as e:
    raise RuntimeError(
        f'Download failed: {e}\n'
    )

if '/workspace' not in sys.path:
    sys.path.insert(0, '/workspace')

import nest_asyncio
nest_asyncio.apply()

from dytopo_engine import DyTopoConfig, run_dytopo, encode, build_adjacency
print('dytopo_engine imported')

Downloaded dytopo_engine.py (23,674 bytes)
dytopo_engine imported


In [4]:
import torch

gpu_count = torch.cuda.device_count()
vram_gb   = torch.cuda.get_device_properties(0).total_memory / 1024**3
total_vram_gb = vram_gb * gpu_count

print(f'GPUs : {gpu_count}x {torch.cuda.get_device_name(0)}')
print(f'VRAM : {vram_gb:.1f} GB per GPU  →  {total_vram_gb:.1f} GB total')

if gpu_count >= 2 and total_vram_gb >= 180:
    VLLM_MODEL   = 'Qwen/Qwen2.5-72B-Instruct'
    VLLM_TP      = 2
    VLLM_MAX_LEN = 8192
    MODEL_LABEL  = 'Qwen2.5-72B bf16 (2× GPU)'
elif vram_gb >= 84:
    VLLM_MODEL   = 'Qwen/Qwen2.5-32B-Instruct'
    VLLM_TP      = 1
    VLLM_MAX_LEN = 8192
    MODEL_LABEL  = 'Qwen2.5-32B bf16'
elif vram_gb >= 48:
    VLLM_MODEL   = 'Qwen/Qwen2.5-32B-Instruct'
    VLLM_TP      = 1
    VLLM_MAX_LEN = 4096
    MODEL_LABEL  = 'Qwen2.5-32B bf16 (reduced ctx)'
elif vram_gb >= 28:
    VLLM_MODEL   = 'Qwen/Qwen2.5-14B-Instruct'
    VLLM_TP      = 1
    VLLM_MAX_LEN = 4096
    MODEL_LABEL  = 'Qwen2.5-14B bf16'
else:
    VLLM_MODEL   = 'Qwen/Qwen2.5-7B-Instruct'
    VLLM_TP      = 1
    VLLM_MAX_LEN = 2048
    MODEL_LABEL  = 'Qwen2.5-7B bf16'

VLLM_PORT = 8000
VLLM_BASE = f'http://localhost:{VLLM_PORT}/v1'

print(f'Selected : {MODEL_LABEL}')
print(f'TP       : {VLLM_TP}')
print(f'Endpoint : {VLLM_BASE}')

GPUs : 2x NVIDIA RTX PRO 6000 Blackwell Workstation Edition
VRAM : 95.1 GB per GPU  →  190.1 GB total
Selected : Qwen2.5-72B bf16 (2× GPU)
TP       : 2
Endpoint : http://localhost:8000/v1


In [ ]:
# ── Cell 6: Wait for vLLM server ──────────────────────────────────────────────
# vLLM must be started from the SSH terminal BEFORE running this cell.
# See README cell at the top for the exact command.
#
# Quick reference (run in SSH terminal):
#   /venv/main/bin/python -m vllm.entrypoints.openai.api_server \
#     --model Qwen/Qwen2.5-72B-Instruct \
#     --max-model-len 8192 \
#     --tensor-parallel-size 2 \
#     --port 8000 \
#     --served-model-name dytopo-model \
#     --trust-remote-code \
#     --disable-log-requests \
#     > /workspace/vllm.log 2>&1 &
#   tail -f /workspace/vllm.log
#
# This cell just polls until the server is healthy (up to 10 min).

import time, httpx

print(f'Waiting for vLLM at http://localhost:{VLLM_PORT}/health ...')
print('Watch live: tail -f /workspace/vllm.log')
print()

for attempt in range(120):
    time.sleep(5)
    try:
        r = httpx.get(f'http://localhost:{VLLM_PORT}/health', timeout=3)
        if r.status_code == 200:
            print(f'✓ vLLM ready after {(attempt+1)*5}s')
            break
    except Exception:
        pass
    if (attempt + 1) % 6 == 0:
        print(f'  ... {(attempt+1)*5}s elapsed')
else:
    raise RuntimeError('vLLM did not become healthy in 10 min — check /workspace/vllm.log')

In [5]:
print('Loading...')

from datasets import load_dataset
import random, json
print('importing...')

random.seed(42)

print('Loading HumanEval...')
humaneval_ds = load_dataset('openai/openai_humaneval', split='test', trust_remote_code=True)
print(f'{len(humaneval_ds)} problems')

print('Loading MATH-500...')
math500_ds = load_dataset('HuggingFaceH4/MATH-500', split='test')
print(f'{len(math500_ds)} problems')

print('Loading Omni-MATH...')
omnimath_ds = load_dataset('KbsdJames/Omni-MATH', split='test')
print(f'{len(omnimath_ds)} problems')

print('Loading APPS-Competition...')
try:
    apps_ds_all = load_dataset('codeparrot/apps', split='test', trust_remote_code=True)
    apps_ds = apps_ds_all.filter(lambda x: x['difficulty'] == 'competition')
    print(f'{len(apps_ds)} competition problems')
    APPS_OK = True
except Exception as e:
    print(f'{e} — skipping APPS')
    APPS_OK = False

Loading...


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'openai/openai_humaneval' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


importing...
Loading HumanEval...


README.md:   0%|          | 0.00/6.52k [00:00<?, ?B/s]

openai_humaneval/test-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 83.9kB            

...

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'codeparrot/apps' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


4428 problems
Loading APPS-Competition...


README.md:   0%|          | 0.00/5.63k [00:00<?, ?B/s]

apps.py:   0%|          | 0.00/4.95k [00:00<?, ?B/s]

Dataset scripts are no longer supported, but found apps.py — skipping APPS


In [6]:
import pandas as pd

# HumanEval
humaneval_problems = [
    {'id': x['task_id'], 'problem': x['prompt'], 'canonical': x['canonical_solution'],
     'test': x['test'], 'entry_point': x['entry_point'], 'dataset': 'humaneval'}
    for x in humaneval_ds
]

# APPS-Competition
if APPS_OK:
    apps_idxs = random.sample(range(len(apps_ds)), 100)
    apps_problems = [
        {'id': f'apps_{i}', 'problem': apps_ds[i]['question'],
         'test_cases': apps_ds[i]['input_output'],
         'solutions': apps_ds[i]['solutions'], 'dataset': 'apps'}
        for i in apps_idxs
    ]
else:
    apps_problems = []

# MATH-500 — all 500
math500_problems = [
    {'id': f'math500_{i}', 'problem': x['problem'], 'answer': x['answer'],
     'subject': x['subject'], 'level': x['level'], 'dataset': 'math500'}
    for i, x in enumerate(math500_ds)
]

# Omni-MATH
def sample_omnimath(ds, n_domains=7, n_per_domain=10, seed=42):
    rng = random.Random(seed)
    dom_key = 'domain' if 'domain' in ds.column_names else 'subject'
    by_dom = {}
    for i, x in enumerate(ds):
        raw = x.get(dom_key, 'unknown')
        d = (raw[0] if isinstance(raw, list) and raw else str(raw)).split('/')[0].strip()
        by_dom.setdefault(d, []).append((i, x))
    top = sorted(by_dom, key=lambda k: -len(by_dom[k]))[:n_domains]
    print(f'  Omni-MATH domains: {top}')
    out = []
    for dom in top:
        for idx, x in rng.sample(by_dom[dom], min(n_per_domain, len(by_dom[dom]))):
            out.append({'id': f'omnimath_{idx}',
                        'problem': x.get('problem', x.get('question', '')),
                        'answer': x.get('answer', x.get('solution', '')),
                        'domain': dom, 'dataset': 'omnimath'})
    return out

omnimath_problems = sample_omnimath(omnimath_ds)

print(f'\nProblem counts:')
for name, probs in [('HumanEval', humaneval_problems), ('APPS', apps_problems),
                     ('MATH-500', math500_problems), ('Omni-MATH', omnimath_problems)]:
    print(f'  {name:18s}: {len(probs)}')
print(f'  TOTAL: {len(humaneval_problems)+len(apps_problems)+len(math500_problems)+len(omnimath_problems)}')

  Omni-MATH domains: ['Mathematics -> Discrete Mathematics -> Combinatorics', 'Mathematics -> Applied Mathematics -> Statistics -> Probability -> Counting Methods -> Combinations', 'Mathematics -> Algebra -> Algebra -> Equations and Inequalities', 'Mathematics -> Geometry -> Plane Geometry -> Polygons', 'Mathematics -> Geometry -> Plane Geometry -> Triangulations', 'Mathematics -> Algebra -> Algebra -> Polynomial Operations', 'Mathematics -> Applied Mathematics -> Math Word Problems']

Problem counts:
  HumanEval         : 164
  APPS              : 0
  MATH-500          : 500
  Omni-MATH         : 70
  TOTAL: 734


In [7]:
import re, subprocess, sys, textwrap

def extract_code(text: str) -> str:
    m = re.search(r'```python\s*\n(.+?)```', text, re.DOTALL)
    if m: return m.group(1)
    m = re.search(r'```\s*\n(.+?)```', text, re.DOTALL)
    if m: return m.group(1)
    return text

def eval_humaneval(code: str, problem: dict, timeout: float = 5.0) -> dict:
    script = f'{code}\n\n{problem["test"]}\n\ncheck({problem["entry_point"]})\nprint("PASS")'
    try:
        r = subprocess.run([sys.executable, '-c', script],
                           capture_output=True, text=True, timeout=timeout)
        return {'passed': r.returncode == 0 and 'PASS' in r.stdout,
                'error': r.stderr.strip()[:200] if r.returncode != 0 else None}
    except subprocess.TimeoutExpired:
        return {'passed': False, 'error': f'Timeout {timeout}s'}

def normalize_math(s: str) -> str:
    if not s: return ''
    m = re.search(r'\\boxed\{(.+?)\}', s, re.DOTALL)
    if m: s = m.group(1)
    s = re.sub(r'\\frac\{([^}]+)\}\{([^}]+)\}', r'\1/\2', s)
    s = s.strip().strip('$').strip().rstrip('.')
    return re.sub(r'\s+', '', s).lower()

def eval_math(output: str, ref: str) -> dict:
    mn, rn = normalize_math(output), normalize_math(ref)
    passed = mn == rn
    if not passed:
        nums_m = re.findall(r'-?\d+(?:[./]\d+)?', mn)
        nums_r = re.findall(r'-?\d+(?:[./]\d+)?', rn)
        if nums_m and nums_r and nums_m[-1] == nums_r[-1]:
            passed = True
    return {'passed': passed, 'model_answer': mn, 'ref_answer': rn}

# Sanity checks
assert eval_math(r'\boxed{42}', '42')['passed']
assert eval_math(r'$\frac{3}{4}$', '3/4')['passed']
assert not eval_math('wrong', '5')['passed']
print('Evaluation utilities DONE')

Evaluation utilities DONE


In [8]:
# ── Core evaluation loop ─────────────────────────────────────────────
import asyncio, json, time
from pathlib import Path
from tqdm.notebook import tqdm

async def run_full_dataset(
    problems: list,
    task_type: str,
    results_path: Path,
    tau: float = 0.3,
    max_rounds: int = 5,
    concurrency: int = 2,
    problem_timeout: float = 180.0,
):
    all_results = []
    done_ids = set()
    if results_path.exists():
        with open(results_path) as f:
            all_results = json.load(f)
        done_ids = {r['id'] for r in all_results}
        print(f'  Resuming: {len(done_ids)}/{len(problems)} already done')

    cfg = DyTopoConfig(
        task_type=task_type,
        n_agents=4,
        tau=tau,
        max_rounds=max_rounds,
        mock_mode=False,
        llm_base_url=VLLM_BASE,
        llm_model='dytopo-model',
        temperature=0.3,
        max_tokens=1200,
    )

    remaining = [p for p in problems if p['id'] not in done_ids]
    sem = asyncio.Semaphore(concurrency)

    async def run_one(prob):
        async with sem:
            t0 = time.monotonic()
            try:
                rounds = []

                async def _collect():
                    async for rr in run_dytopo(prob['problem'], cfg):
                        rounds.append(rr.to_dict())

                await asyncio.wait_for(_collect(), timeout=problem_timeout)

                final = next(
                    (r['final_answer'] for r in reversed(rounds) if r.get('final_answer')), ''
                )
                return {
                    'id': prob['id'], 'dataset': prob['dataset'],
                    'n_rounds': len(rounds), 'final_answer': final,
                    'time_s': round(time.monotonic() - t0, 2),
                    'adjacency_per_round': [r['adjacency'] for r in rounds],
                }
            except asyncio.TimeoutError:
                return {'id': prob['id'], 'dataset': prob['dataset'],
                        'error': f'Timeout after {problem_timeout}s',
                        'time_s': round(time.monotonic() - t0, 2)}
            except Exception as e:
                return {'id': prob['id'], 'dataset': prob['dataset'],
                        'error': f'{type(e).__name__}: {e}',
                        'time_s': round(time.monotonic() - t0, 2)}

    # BATCH=1: save + update tqdm after every single problem so progress is visible
    BATCH = 1
    pbar = tqdm(total=len(remaining), desc='problems', unit='prob')
    errors = 0
    t_start = time.monotonic()

    for i in range(0, len(remaining), BATCH):
        batch = remaining[i:i+BATCH]
        batch_results = await asyncio.gather(*[run_one(p) for p in batch])

        for r in batch_results:
            all_results.append(r)
            if 'error' in r:
                errors += 1
                pbar.write(f' error {r["id"]}  {r["error"]}')
            else:
                pbar.write(f' ok {r["id"]}  {r["time_s"]:.0f}s  rounds={r["n_rounds"]}')

        with open(results_path, 'w') as f:
            json.dump(all_results, f, indent=2)

        done_so_far = len(all_results) - len(done_ids)
        elapsed = time.monotonic() - t_start
        rate = done_so_far / elapsed if elapsed > 0 else 0
        eta_s = (len(remaining) - done_so_far) / rate if rate > 0 else float('inf')
        eta_str = f'{eta_s/3600:.1f}h' if eta_s < 86400 else '?'

        pbar.update(len(batch))
        pbar.set_postfix(saved=len(all_results), errors=errors, rate=f'{rate:.3f}/s', eta=eta_str)

    pbar.close()
    total_errors = sum(1 for r in all_results if 'error' in r)
    print(f'\n  Done: {len(all_results)} results  |  errors: {total_errors}  |  saved to {results_path}')
    return all_results

print('Evaluation loop ready (batch=1, timeout=180s, concurrency=3)')

Evaluation loop ready (batch=1, timeout=180s, concurrency=3)


In [ ]:
import time

async def estimate_per_problem(task_type='code'):
    cfg = DyTopoConfig(
        task_type=task_type, n_agents=4, tau=0.3, max_rounds=3,
        mock_mode=False, llm_base_url=VLLM_BASE, llm_model='dytopo-model',
        temperature=0.3, max_tokens=400,
    )
    t0 = time.monotonic()
    async for _ in run_dytopo('Write a Python function to check if a number is prime.', cfg):
        pass
    return time.monotonic() - t0

print('Timing one code problem...')
t_code = await estimate_per_problem('code')
print(f'  Code: {t_code:.1f}s/problem')

print('Timing one math problem...')
t_math = await estimate_per_problem('math')
print(f'  Math: {t_math:.1f}s/problem')

concurrency = 2  # match default in run_full_dataset
estimates = [
    ('HumanEval', len(humaneval_problems), t_code),
    ('MATH-500',  len(math500_problems),   t_math),
    ('Omni-MATH', len(omnimath_problems),  t_math * 1.3),
]
if APPS_OK:
    estimates.insert(1, ('APPS', len(apps_problems), t_code * 1.5))

print(f'\nRuntime estimates (concurrency={concurrency}):')
total_s = 0
for name, n, t_per in estimates:
    est = (n * t_per) / concurrency
    total_s += est
    h, m = divmod(int(est), 3600); m //= 60
    print(f'  {name:12s}: {n} probs × {t_per:.0f}s / {concurrency} = {h}h{m:02d}m')

th, tm = divmod(int(total_s), 3600); tm //= 60
cost = total_s / 3600 * 2.691
print(f'\n  TOTAL : {th}h{tm:02d}m  (~${cost:.2f} at $2.691/hr)')

In [ ]:
# ── RUN FULL EVALUATION

DATASETS_TO_RUN = [
    # (name,       problems,           task_type, tau,  max_rounds)
    ('humaneval',  humaneval_problems, 'code',    0.3,  5),
    ('math500',    math500_problems,   'math',    0.3,  5),
    ('omnimath',   omnimath_problems,  'math',    0.4,  5),
]
if APPS_OK:
    DATASETS_TO_RUN.append(('apps', apps_problems, 'code', 0.3, 5))

all_run_results = {}

for ds_name, problems, task_type, tau, max_rounds in DATASETS_TO_RUN:
    results_path = RESULTS_DIR / f'{ds_name}_results.json'
    print(f'\n═══ {ds_name.upper()} ({len(problems)} problems, τ={tau}) ═══')
    results = await run_full_dataset(
        problems, task_type, results_path,
        tau=tau, max_rounds=max_rounds, concurrency=3,
    )
    all_run_results[ds_name] = results
    print(f'  Done: {len(results)} results saved to {results_path}')



═══ HUMANEVAL (164 problems, τ=0.3) ═══
  Resuming: 164/164 already done


problems: 0prob [00:00, ?prob/s]


  Done: 164 results  |  errors: 0  |  saved to /workspace/results_v2/humaneval_results.json
  Done: 164 results saved to /workspace/results_v2/humaneval_results.json

═══ MATH500 (500 problems, τ=0.3) ═══
  Resuming: 365/500 already done


problems:   0%|          | 0/135 [00:00<?, ?prob/s]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

 ok math500_365  103s  rounds=5
 ok math500_366  62s  rounds=5
 ok math500_367  24s  rounds=2
 ok math500_368  58s  rounds=5
 ok math500_369  45s  rounds=4
 ok math500_370  55s  rounds=4
 ok math500_371  85s  rounds=5
 ok math500_372  89s  rounds=5
 ok math500_373  65s  rounds=5
 ok math500_374  26s  rounds=3
 ok math500_375  91s  rounds=5
 ok math500_376  71s  rounds=5
 ok math500_377  60s  rounds=5
 ok math500_378  71s  rounds=5
 ok math500_379  67s  rounds=5
 ok math500_380  65s  rounds=5
 ok math500_381  56s  rounds=4
 ok math500_382  59s  rounds=4
 ok math500_383  60s  rounds=5
 ok math500_384  34s  rounds=4
 ok math500_385  43s  rounds=4
 ok math500_386  59s  rounds=5
 ok math500_387  50s  rounds=5
 ok math500_388  67s  rounds=5
 ok math500_389  62s  rounds=5
 ok math500_390  102s  rounds=5
 ok math500_391  125s  rounds=5
 ok math500_392  87s  rounds=5
 ok math500_393  72s  rounds=5
 ok math500_394  84s  rounds=5
 ok math500_395  66s  rounds=5
 ok math500_396  57s  rounds=5
 ok m

problems:   0%|          | 0/70 [00:00<?, ?prob/s]

 ok omnimath_937  73s  rounds=5
 ok omnimath_196  76s  rounds=5
 ok omnimath_2202  109s  rounds=5
 ok omnimath_1874  77s  rounds=5
 ok omnimath_1697  33s  rounds=3
 ok omnimath_1089  79s  rounds=5
 ok omnimath_800  72s  rounds=5
 ok omnimath_728  86s  rounds=5
 ok omnimath_4029  65s  rounds=5
 ok omnimath_227  68s  rounds=5
 ok omnimath_454  66s  rounds=5
 ok omnimath_779  84s  rounds=5
 ok omnimath_1212  66s  rounds=5
 ok omnimath_1251  72s  rounds=5
 ok omnimath_2527  80s  rounds=5
 ok omnimath_3327  62s  rounds=5
 ok omnimath_444  127s  rounds=5
 ok omnimath_2992  87s  rounds=5
 ok omnimath_1180  63s  rounds=5
 ok omnimath_2854  79s  rounds=5
 ok omnimath_3658  82s  rounds=5
 ok omnimath_1671  67s  rounds=5
 ok omnimath_3762  79s  rounds=5
 ok omnimath_2181  74s  rounds=5
 ok omnimath_60  89s  rounds=5
 ok omnimath_1238  111s  rounds=5
 ok omnimath_3682  99s  rounds=5
 ok omnimath_2785  93s  rounds=5
 ok omnimath_1217  70s  rounds=5
 ok omnimath_1626  81s  rounds=5
 ok omnimath_4289

ModuleNotFoundError: No module named 'jsonhow'

In [ ]:
import json
from pathlib import Path

RESULTS_DIR = Path('/workspace/results_v2')

all_run_results = {}
for ds_name in ['humaneval', 'math500', 'omnimath', 'apps']:
    p = RESULTS_DIR / f'{ds_name}_results.json'
    if p.exists():
        with open(p) as f:
            all_run_results[ds_name] = json.load(f)
        print(f'  {ds_name:12s}: {len(all_run_results[ds_name])} results loaded')
    else:
        print(f'  {ds_name:12s}: not found — skipping')

print(f'\nLoaded {len(all_run_results)} datasets from {RESULTS_DIR}')

# ── Score ──────────────────────────────────────────────────────────────────────
prob_maps = {
    'humaneval': {p['id']: p for p in humaneval_problems},
    'math500':   {p['id']: p for p in math500_problems},
    'omnimath':  {p['id']: p for p in omnimath_problems},
    'apps':      {p['id']: p for p in apps_problems},
}

def score_dataset(results, ds_name):
    prob_map = prob_maps[ds_name]
    passed = errors = 0
    for r in results:
        if 'error' in r:
            errors += 1; continue
        prob = prob_map.get(r['id'])
        if not prob: continue
        if ds_name == 'humaneval':
            code = extract_code(r.get('final_answer', ''))
            ok = eval_humaneval(code, prob)['passed']
        else:
            ok = eval_math(r.get('final_answer', ''), prob.get('answer', ''))['passed']
        if ok: passed += 1
    total = len(results)
    return {'dataset': ds_name, 'total': total, 'passed': passed,
            'errors': errors, 'accuracy': passed / total * 100 if total else 0}

print()
scores = {}
for ds_name, results in all_run_results.items():
    s = score_dataset(results, ds_name)
    scores[ds_name] = s
    print(f'{ds_name:18s}: {s["passed"]}/{s["total"]} ({s["accuracy"]:.1f}%)  errors={s["errors"]}')
# ── Compare to paper Table 2 ─────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

paper_data = {
    'HumanEval': {
        'single_agent': 18.29, 'random_topo': 80.49,
        'agentscope': 80.00, 'dytopo_paper': 89.63,
    },
    'APPS-Comp': {
        'single_agent': 9.62,  'random_topo': 24.24,
        'agentscope': 23.12, 'dytopo_paper': 25.14,
    },
    'MATH-500': {
        'single_agent': 48.57, 'random_topo': 68.57,
        'agentscope': 72.86, 'dytopo_paper': 75.71,
    },
    'Omni-MATH': {
        'single_agent': 21.43, 'random_topo': 25.71,
        'agentscope': 35.71, 'dytopo_paper': 51.43,
    },
}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.patch.set_facecolor('#0a0e1a')

ds_labels = list(paper_data.keys())
ds_map_local = {'HumanEval': 'humaneval', 'APPS-Comp': 'apps',
                'MATH-500': 'math500', 'Omni-MATH': 'omnimath'}
x = np.arange(len(ds_labels))
width = 0.18

METHOD_COLORS = {
    'single_agent':  '#475569',
    'random_topo':   '#f87171',
    'agentscope':    '#fbbf24',
    'dytopo_paper':  '#4ade80',
    'dytopo_local':  '#38bdf8',
}
METHOD_LABELS = {
    'single_agent':  'Single Agent',
    'random_topo':   'Random Topology',
    'agentscope':    'AgentScope',
    'dytopo_paper':  'DyTopo paper (Qwen3-8B)',
    'dytopo_local':  f'DyTopo local ({MODEL_LABEL})',
}

ax = axes[0]
ax.set_facecolor('#0f1828')
methods = ['single_agent', 'random_topo', 'agentscope', 'dytopo_paper']
for i, m in enumerate(methods):
    vals = [paper_data[d][m] for d in ds_labels]
    ax.bar(x + (i - 1.5) * width, vals, width,
           label=METHOD_LABELS[m], color=METHOD_COLORS[m], alpha=0.85)
local_vals = [scores.get(ds_map_local[d], {}).get('accuracy') for d in ds_labels]
if any(v is not None for v in local_vals):
    ax.scatter(x, [v or 0 for v in local_vals], marker='*', s=300,
               color=METHOD_COLORS['dytopo_local'], zorder=5,
               label=METHOD_LABELS['dytopo_local'])
ax.set_xticks(x); ax.set_xticklabels(ds_labels, color='white', fontsize=9)
ax.set_ylabel('Accuracy (%)', color='white')
ax.set_title('DyTopo vs Baselines (bars=paper, stars=local)', color='white', fontsize=10)
ax.legend(facecolor='#1a2235', edgecolor='#1e293b', labelcolor='white', fontsize=8)
ax.tick_params(colors='white'); ax.spines[:].set_color('#1e293b'); ax.grid(axis='y', alpha=0.2)

ax2 = axes[1]
ax2.set_facecolor('#0f1828')
for i, m in enumerate(['random_topo', 'agentscope', 'dytopo_paper']):
    improvements = [paper_data[d][m] - paper_data[d]['single_agent'] for d in ds_labels]
    ax2.bar(x + (i - 1) * width, improvements, width,
            label=METHOD_LABELS[m], color=METHOD_COLORS[m], alpha=0.85)
if any(v is not None for v in local_vals):
    local_imp = [
        (scores.get(ds_map_local[d], {}).get('accuracy') or 0) - paper_data[d]['single_agent']
        for d in ds_labels
    ]
    ax2.scatter(x, local_imp, marker='*', s=300,
                color=METHOD_COLORS['dytopo_local'], zorder=5,
                label=METHOD_LABELS['dytopo_local'])
ax2.axhline(0, color='#94a3b8', linewidth=1, linestyle='--')
ax2.set_xticks(x); ax2.set_xticklabels(ds_labels, color='white', fontsize=9)
ax2.set_ylabel('Improvement over Single Agent (pp)', color='white')
ax2.set_title('Gain from Multi-Agent Routing', color='white', fontsize=10)
ax2.legend(facecolor='#1a2235', edgecolor='#1e293b', labelcolor='white', fontsize=8)
ax2.tick_params(colors='white'); ax2.spines[:].set_color('#1e293b'); ax2.grid(axis='y', alpha=0.2)

plt.tight_layout()
out_path = '/workspace/dytopo_results_comparison.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='#0a0e1a')
import shutil
shutil.copy(out_path, RESULTS_DIR / 'dytopo_results_comparison.png')
plt.show()
print(f'Saved to {out_path}')

ModuleNotFoundError: No module named 'jsonhow'

In [ ]:
# ── Cell 16: Edge density analysis across real problems ───────────────────────
import matplotlib.pyplot as plt, shutil

def compute_density(adj, n):
    if n < 2: return 0.0
    edges = sum(adj[j][i] for j in range(n) for i in range(n) if j != i)
    return edges / (n * (n - 1))

round_densities_by_dataset = {}
for ds_name, results in all_run_results.items():
    by_round = {}
    for r in results:
        for round_idx, adj_matrix in enumerate(r.get('adjacency_per_round', [])):
            n = len(adj_matrix)
            d = compute_density(adj_matrix, n)
            by_round.setdefault(round_idx, []).append(d)
    round_densities_by_dataset[ds_name] = {
        k: sum(v)/len(v) for k, v in by_round.items()
    }

COLORS_DS = {'humaneval': '#38bdf8', 'apps': '#f87171', 'math500': '#fbbf24', 'omnimath': '#a78bfa'}

fig, ax = plt.subplots(figsize=(9, 5))
ax.set_facecolor('#0f1828')
fig.patch.set_facecolor('#0a0e1a')

for ds_name, by_round in round_densities_by_dataset.items():
    if not by_round: continue
    rounds_x = sorted(by_round.keys())
    densities = [by_round[r] for r in rounds_x]
    ax.plot([r+1 for r in rounds_x], densities, 'o-',
            color=COLORS_DS.get(ds_name, '#94a3b8'), linewidth=2.5, markersize=8,
            label=ds_name)

ax.set_xlabel('Round', color='white', fontsize=11)
ax.set_ylabel('Avg Edge Density', color='white', fontsize=11)
ax.set_title('Topology Evolution on Real Datasets (Real LLM)\n'
             'Confirms dense exploration → sparse finalization pattern from paper Fig 4',
             color='white', fontsize=10)
ax.legend(facecolor='#1a2235', edgecolor='#1e293b', labelcolor='white', fontsize=9)
ax.tick_params(colors='white'); ax.spines[:].set_color('#1e293b'); ax.grid(alpha=0.2)
ax.set_ylim(0, 1.05)

plt.tight_layout()
out_path = '/workspace/dytopo_real_density_evolution.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='#0a0e1a')
shutil.copy(out_path, RESULTS_DIR / 'dytopo_real_density_evolution.png')
plt.show()

In [ ]:
# ── Cell 17: Full results summary ─────────────────────────────────────────────
print('╔══════════════════════════════════════════════════════════════════════╗')
print('║            DyTopo Reproduction Results (Vast.ai)                    ║')
print('╠══════════════════════════════════════════════════════════════════════╣')
print(f'║  Model: {MODEL_LABEL:<60}║')
print('╠═══════════════════╦═══════════╦═══════════╦═══════════╦═════════════╣')
print('║ Dataset           ║ Our score ║ Paper Dyt ║ Paper SA  ║ Improvement ║')
print('╠═══════════════════╬═══════════╬═══════════╬═══════════╬═════════════╣')

paper_dytopo = {'humaneval': 89.63, 'apps': 25.14, 'math500': 75.71, 'omnimath': 51.43}
paper_single = {'humaneval': 18.29, 'apps': 9.62,  'math500': 48.57, 'omnimath': 21.43}

for ds in ['humaneval', 'apps', 'math500', 'omnimath']:
    s = scores.get(ds)
    if s:
        our   = f'{s["accuracy"]:.1f}%'
        paper = f'{paper_dytopo.get(ds, 0):.1f}%'
        sa    = f'{paper_single.get(ds, 0):.1f}%'
        diff  = s['accuracy'] - paper_single.get(ds, 0)
        imp   = f'+{diff:.1f}pp' if diff >= 0 else f'{diff:.1f}pp'
        print(f'║ {ds:17s} ║ {our:>9s} ║ {paper:>9s} ║ {sa:>9s} ║ {imp:>11s} ║')
    else:
        print(f'║ {ds:17s} ║  not run  ║ {paper_dytopo.get(ds,0):>7.1f}% ║           ║             ║')

print('╚═══════════════════╩═══════════╩═══════════╩═══════════╩═════════════╝')
print()
print('All results saved to:', str(RESULTS_DIR))